# NCF Training Notebook — Kaggle GPU
### Hybrid NeuMF with Tags + Genres + Timestamps
**Fixes applied:**
- `num_workers=4` + `pin_memory=True` → eliminates CPU-GPU bottleneck
- `DataParallel` → uses BOTH T4 GPUs if available
- Trains on `train_ratings.csv` (honest split, not full data)
- Larger batch size (`8192`) for GPU efficiency

In [ ]:
import torch, sklearn, tqdm, joblib
print(f"✅ PyTorch     : {torch.__version__}")
print(f"✅ CUDA        : {torch.cuda.is_available()} ({torch.cuda.device_count()} GPUs)")
print(f"✅ scikit-learn: {sklearn.__version__}")
print(f"✅ tqdm        : {tqdm.__version__}")
print(f"✅ joblib      : {joblib.__version__}")
print()
print("scikit-surprise: NOT needed for NCF ✅")


In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
print(f"PyTorch version: {torch.__version__}")

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import joblib
import logging
from pathlib import Path
import random
from sklearn.preprocessing import MultiLabelBinarizer, MinMaxScaler
import os

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

# --- KAGGLE DYNAMIC PATH FINDING ---
# Scans /kaggle/input for our filtered CSV files
KAGGLE_ROOT = "/kaggle/input"
rating_path, tag_path, movie_path, train_rating_path = None, None, None, None

for root, _, files in os.walk(KAGGLE_ROOT):
    for f in files:
        fp = os.path.join(root, f)
        if f.lower() == "train_ratings.csv":   # preferred: honest train split
            train_rating_path = fp
        elif f.lower() == "rating.csv":         # fallback: full filtered ratings
            rating_path = fp
        elif f.lower() == "tag.csv":
            tag_path = fp
        elif f.lower() == "movie.csv":
            movie_path = fp

# Use train split if available, otherwise fall back to full rating.csv
RATINGS_DIR = Path(train_rating_path or rating_path)
if not RATINGS_DIR.exists():
    raise FileNotFoundError("Could not find rating.csv or train_ratings.csv! Attach the dataset in the Kaggle UI.")
TAGS_DIR    = Path(tag_path)
MOVIES_CSV_PATH = Path(movie_path)

# Kaggle output directory
NCF_MODEL_PATH = Path("/kaggle/working/ncf_model")
NCF_MODEL_PATH.mkdir(parents=True, exist_ok=True)

logger.info(f"Ratings file : {RATINGS_DIR}")
logger.info(f"Tags file    : {TAGS_DIR}")
logger.info(f"Movies file  : {MOVIES_CSV_PATH}")
logger.info(f"Output dir   : {NCF_MODEL_PATH}")

In [ ]:
# ───────────────────────────────────────────
# HYBRID IMPLICIT DATASET
# ───────────────────────────────────────────
class HybridImplicitDataset(Dataset):
    def __init__(self, users, items, timestamps, labels, movie_genre_matrix, movie_tags_matrix):
        self.users = torch.tensor(users, dtype=torch.long)
        self.items = torch.tensor(items, dtype=torch.long)
        self.timestamps = torch.tensor(timestamps, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.float32)
        self.movie_genre_matrix = torch.tensor(movie_genre_matrix, dtype=torch.float32)
        self.movie_tags_matrix  = torch.tensor(movie_tags_matrix,  dtype=torch.long)

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        u = self.users[idx]
        i = self.items[idx]
        return u, i, self.movie_genre_matrix[i], self.timestamps[idx], self.movie_tags_matrix[i], self.labels[idx]


# ───────────────────────────────────────────
# HYBRID NeuMF MODEL
# ───────────────────────────────────────────
class HybridNeuMFModel(nn.Module):
    def __init__(self, num_users, num_items, num_genres, num_tags,
                 mf_dim=16, mlp_dim=32, tag_dim=16, hidden_layers=[128, 64, 32]):
        super(HybridNeuMFModel, self).__init__()

        self.embedding_user_mf  = nn.Embedding(num_users, mf_dim)
        self.embedding_item_mf  = nn.Embedding(num_items, mf_dim)
        self.embedding_user_mlp = nn.Embedding(num_users, mlp_dim)
        self.embedding_item_mlp = nn.Embedding(num_items, mlp_dim)
        self.tag_embedding      = nn.EmbeddingBag(num_tags, tag_dim, mode='mean', padding_idx=0)

        self.genre_layer = nn.Sequential(nn.Linear(num_genres, 16), nn.ReLU())

        input_dim = mlp_dim * 2 + 16 + tag_dim + 1
        layers = []
        for hidden_dim in hidden_layers:
            layers += [nn.Linear(input_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(), nn.Dropout(0.2)]
            input_dim = hidden_dim
        self.mlp = nn.Sequential(*layers)

        final_dim = mf_dim + hidden_layers[-1]
        self.prediction_layer = nn.Sequential(nn.Linear(final_dim, 1), nn.Sigmoid())
        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.embedding_user_mf.weight,  std=0.01)
        nn.init.normal_(self.embedding_item_mf.weight,  std=0.01)
        nn.init.normal_(self.embedding_user_mlp.weight, std=0.01)
        nn.init.normal_(self.embedding_item_mlp.weight, std=0.01)
        for m in self.mlp:
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
        nn.init.kaiming_uniform_(self.prediction_layer[0].weight, nonlinearity='sigmoid')

    def forward(self, user_indices, item_indices, genres, timestamps, tags):
        mf_vector  = torch.mul(self.embedding_user_mf(user_indices), self.embedding_item_mf(item_indices))
        tag_vector  = self.tag_embedding(tags)
        genre_vector = self.genre_layer(genres)
        time_vector  = timestamps.unsqueeze(1)
        mlp_vector   = torch.cat([self.embedding_user_mlp(user_indices),
                                   self.embedding_item_mlp(item_indices),
                                   genre_vector, time_vector, tag_vector], dim=1)
        mlp_vector = self.mlp(mlp_vector)
        return self.prediction_layer(torch.cat([mf_vector, mlp_vector], dim=1)).squeeze(1)


# ───────────────────────────────────────────
# RECOMMENDER WRAPPER
# ───────────────────────────────────────────
class NeuralCollaborativeRecommender:
    def __init__(self, mf_dim=16, mlp_dim=32, tag_dim=16, n_epochs=5, batch_size=8192, lr=0.001):
        self.mf_dim, self.mlp_dim, self.tag_dim = mf_dim, mlp_dim, tag_dim
        self.n_epochs, self.batch_size, self.lr  = n_epochs, batch_size, lr
        self.model = None
        self.movies_df = None
        self.user2idx, self.idx2user = {}, {}
        self.item2idx, self.idx2item = {}, {}
        self.tag2idx   = {"<PAD>": 0}
        self.idx2tag   = {0: "<PAD>"}
        self.movie_tags = {}
        self.max_tags_per_movie = 20
        self.mlb         = MultiLabelBinarizer()
        self.time_scaler = MinMaxScaler()

        # Use CUDA on Kaggle, MPS on Mac, CPU fallback
        if torch.cuda.is_available():
            self.device = torch.device("cuda")
        elif torch.backends.mps.is_available():
            self.device = torch.device("mps")
        else:
            self.device = torch.device("cpu")
        logger.info(f"Training device: {self.device}")

    def _prepare_movie_tags(self):
        logger.info("Parsing tags for Bag Processing...")
        if not TAGS_DIR.exists():
            return
        tags_df = pd.read_csv(TAGS_DIR)
        tags_df['tag'] = tags_df['tag'].astype(str).str.lower().str.strip()
        counts    = tags_df['tag'].value_counts()
        valid_tags = counts[counts >= 3].index
        tags_df   = tags_df[tags_df['tag'].isin(valid_tags)]
        for idx, tag in enumerate(tags_df['tag'].unique(), start=1):
            self.tag2idx[tag] = idx
            self.idx2tag[idx] = tag
        logger.info(f"Loaded {len(self.tag2idx)} unique valid tags.")
        grouped = tags_df.groupby('movieId')['tag'].apply(list).reset_index()
        for _, row in grouped.iterrows():
            m_id     = row['movieId']
            int_list = list(set(self.tag2idx[w] for w in row['tag'] if w in self.tag2idx))
            int_list = int_list[:self.max_tags_per_movie]
            self.movie_tags[m_id] = int_list + [0] * (self.max_tags_per_movie - len(int_list))

    def fit(self, train_df):
        logger.info(f"Training on {len(train_df):,} ratings...")
        movies_df  = pd.read_csv(MOVIES_CSV_PATH)
        self.movies_df = movies_df
        self._prepare_movie_tags()

        merged = train_df.merge(movies_df[['movieId', 'genres']], on='movieId', how='left')
        merged['genres'] = merged['genres'].fillna("Unknown").str.split('|')
        self.mlb.fit(merged['genres'])

        if 'timestamp' in train_df.columns:
            numeric_ts = pd.to_numeric(train_df['timestamp'], errors='coerce').fillna(0).astype(np.float32)
            self.time_scaler.fit(numeric_ts.values.reshape(-1, 1))

        all_users = train_df['userId'].astype(str).unique()
        all_items = train_df['movieId'].astype(int).unique()
        self.user2idx = {str(u): i for i, u in enumerate(all_users)}
        self.idx2user = {idx: uid for uid, idx in self.user2idx.items()}
        self.item2idx = {int(i): idx for idx, i in enumerate(all_items)}
        self.idx2item = {idx: iid for iid, idx in self.item2idx.items()}

        user_interacted = train_df.groupby('userId')['movieId'].apply(
            lambda x: set(x.astype(int).map(self.item2idx))
        ).to_dict()

        n_train = len(train_df)
        n_total = n_train * 5
        u_arr = np.empty(n_total, dtype=np.int32)
        i_arr = np.empty(n_total, dtype=np.int32)
        y_arr = np.empty(n_total, dtype=np.float32)

        users_mapped = train_df['userId'].astype(str).map(self.user2idx).values
        items_mapped = train_df['movieId'].astype(int).map(self.item2idx).values
        u_arr[:n_train] = users_mapped
        i_arr[:n_train] = items_mapped
        y_arr[:n_train] = 1.0

        num_items = len(all_items)
        random_negatives   = np.random.randint(0, num_items, size=(n_train, 4))
        user_ids_original  = train_df['userId'].values

        logger.info("Building negative samples (collision-free)...")
        from tqdm import tqdm
        idx_offset = n_train
        for i in tqdm(range(n_train), desc="Negative Sampling"):
            interacted = user_interacted[user_ids_original[i]]
            negs = random_negatives[i]
            while any(n in interacted for n in negs):
                negs = np.random.randint(0, num_items, size=4)
            u_arr[idx_offset:idx_offset+4] = users_mapped[i]
            i_arr[idx_offset:idx_offset+4] = negs
            y_arr[idx_offset:idx_offset+4] = 0.0
            idx_offset += 4

        num_genres = len(self.mlb.classes_)
        movies_df['genres_split'] = movies_df['genres'].fillna("Unknown").str.split('|')
        all_genres_encoded = self.mlb.transform(movies_df['genres_split'])
        m_id_to_genre = {m_id: all_genres_encoded[i] for i, m_id in enumerate(movies_df['movieId'].values)}

        movie_genre_matrix = np.zeros((num_items, num_genres), dtype=np.float32)
        movie_tags_matrix  = np.zeros((num_items, self.max_tags_per_movie), dtype=np.int32)
        default_pad = [0] * self.max_tags_per_movie
        for item_idx, m_id in self.idx2item.items():
            movie_tags_matrix[item_idx]  = self.movie_tags.get(m_id, default_pad)
            movie_genre_matrix[item_idx] = m_id_to_genre.get(m_id, np.zeros(num_genres, dtype=np.float32))

        if 'timestamp' in train_df.columns:
            numeric_ts_pos = pd.to_numeric(train_df['timestamp'], errors='coerce').fillna(0).astype(np.float32)
            t_scaled = self.time_scaler.transform(numeric_ts_pos.values.reshape(-1, 1)).flatten()
        else:
            t_scaled = np.zeros(n_train)
        t_arr = np.zeros(n_total, dtype=np.float32)
        t_arr[:n_train] = t_scaled

        train_dataset = HybridImplicitDataset(u_arr, i_arr, t_arr, y_arr, movie_genre_matrix, movie_tags_matrix)

        # ── KEY FIX: num_workers eliminates CPU-GPU bottleneck ──
        n_workers = min(4, os.cpu_count() or 1)
        train_loader = DataLoader(
            train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=n_workers,    # parallel CPU workers → GPU never idles
            pin_memory=True,          # fast CPU→GPU memory transfer
            persistent_workers=True   # workers stay alive between epochs
        )
        logger.info(f"DataLoader: batch_size={self.batch_size}, num_workers={n_workers}, pin_memory=True")

        base_model = HybridNeuMFModel(
            len(all_users), len(all_items), num_genres, len(self.tag2idx),
            self.mf_dim, self.mlp_dim, self.tag_dim
        ).to(self.device)

        # ── KEY FIX: DataParallel uses BOTH T4 GPUs on Kaggle ──
        if torch.cuda.device_count() > 1:
            logger.info(f"Using DataParallel across {torch.cuda.device_count()} GPUs!")
            self.model = nn.DataParallel(base_model)
        else:
            self.model = base_model

        criterion = nn.BCELoss()
        optimizer = torch.optim.Adam(self.model.parameters(), lr=self.lr)

        import time
        logger.info(f"Training {len(u_arr):,} tensor rows on {self.device} ...")
        start_time = time.time()
        for epoch in range(self.n_epochs):
            self.model.train()
            total_loss = 0
            for batch_u, batch_i, batch_g, batch_t, batch_tg, batch_y in tqdm(train_loader, desc=f"Epoch {epoch+1}/{self.n_epochs}"):
                batch_u  = batch_u.to(self.device,  non_blocking=True)
                batch_i  = batch_i.to(self.device,  non_blocking=True)
                batch_g  = batch_g.to(self.device,  non_blocking=True)
                batch_t  = batch_t.to(self.device,  non_blocking=True)
                batch_tg = batch_tg.to(self.device, non_blocking=True)
                batch_y  = batch_y.to(self.device,  non_blocking=True)
                optimizer.zero_grad()
                preds = self.model(batch_u, batch_i, batch_g, batch_t, batch_tg)
                loss  = criterion(preds, batch_y)
                loss.backward()
                optimizer.step()
                total_loss += loss.item() * len(batch_u)
            logger.info(f"Epoch {epoch+1}/{self.n_epochs} — BCE Loss: {total_loss/len(train_dataset):.4f}")
        logger.info(f"Total training time: {(time.time()-start_time)/60:.1f} minutes")

        # Unwrap DataParallel before saving so weights load cleanly on any device
        self.model = self.model.module if hasattr(self.model, 'module') else self.model

    def save(self):
        if self.model is None:
            raise ValueError("Model not trained.")
        NCF_MODEL_PATH.mkdir(parents=True, exist_ok=True)
        mappings = {
            'user2idx': self.user2idx, 'idx2user': self.idx2user,
            'item2idx': self.item2idx, 'idx2item': self.idx2item,
            'mf_dim': self.mf_dim, 'mlp_dim': self.mlp_dim, 'tag_dim': self.tag_dim,
            'mlb': self.mlb, 'time_scaler': self.time_scaler,
            'tag2idx': self.tag2idx, 'idx2tag': self.idx2tag, 'movie_tags': self.movie_tags
        }
        joblib.dump(mappings, NCF_MODEL_PATH / "neumf_mappings.pkl")
        torch.save(self.model.state_dict(), NCF_MODEL_PATH / "neumf_weights.pt")
        logger.info(f"Model saved to {NCF_MODEL_PATH}")

In [ ]:
if __name__ == "__main__" or True:  # always run in notebook context
    logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
    logger.info("Loading ratings...")
    ratings = pd.read_csv(RATINGS_DIR)
    logger.info(f"Loaded {len(ratings):,} ratings from {RATINGS_DIR.name}")

    recommender = NeuralCollaborativeRecommender(
        n_epochs=5,
        batch_size=8192   # larger batch = better GPU utilization
    )
    recommender.fit(ratings)
    recommender.save()
    logger.info("Training complete! Download neumf_weights.pt and neumf_mappings.pkl from /kaggle/working/ncf_model/")